# Phase C v3: A/B/D 3条件アブレーション — 入力形式の実験的確定

## Force is Oblivion / Hegemonikón Research (Lēthē)

**目的**: Phase C の入力形式を実験的に確定する。化学的同型に基づく U⊣N 随伴の方向問題。

| 条件 | 入力 | U⊣N 方向 | 化学比喩 |
|:-----|:-----|:---------|:---------|
| **A** | CCL テキストのみ | N 方向 (CCL→表現) | SMILES 記法 |
| **B** | Code + CCL 並置 | 両方向同時 | 構造式+分子名併記 |
| **D** | Code のみ | U 方向 (Code→構造) | X線結晶構造解析 |

### v3 修正点 (欠陥対応)
- **ρ_ccl**: ccl_edit_dist との相関で Phase C が 49d を超えるか測定
- **MAX_LEN 条件別**: A/D=512, B=1024 (truncation 不公平を解消)
- **R@k 修正**: fold 内 retrieval metrics (正例 recall in top-k)
- **診断ペア**: 構造異性体 + 49d 盲点による事後評価

### ベースライン
- Phase C-mini (CodeBERT 125M, 246ペア): ρ=0.963, 偏ρ=0.960

### 仮説
| # | 仮説 | 判定基準 |
|:--|:-----|:---------|
| P42 | A vs B vs D — どの入力形式が最も構造理解に優れるか | max(ρ_ccl) の条件 |
| P11' | 7B QLoRA > Phase C-mini (ρ=0.963) | Δρ > 0 |
| P14 | L_Ξ あり > なし | Δρ > 0 at best λ |

In [ ]:
# Cell 1: 環境セットアップ
!nvidia-smi
!pip install -q transformers peft bitsandbytes accelerate datasets scipy scikit-learn

In [ ]:
# Cell 2: ファイルアップロード
from google.colab import files

print("4ファイルをアップロード:")
print("  1. phase_c_v3.py")
print("  2. phase_c_condition_A_full.jsonl")
print("  3. phase_c_condition_B.jsonl")
print("  4. phase_c_condition_D.jsonl")
uploaded = files.upload()
print(f"\n{len(uploaded)} files uploaded: {list(uploaded.keys())}")

In [ ]:
# Cell 3: Quick テスト (2-fold × 2条件 — 動作確認)
!python phase_c_v3.py --condition A --data phase_c_condition_A_full.jsonl --quick --output quick_test.json

In [ ]:
# Cell 4: 本実行 — 3条件一括 (5-fold × 3λ × 3条件)
# 推定: L4 で 4-8 時間, A100 で 2-4 時間
!python phase_c_v3.py --all --output phase_c_v3_results.json

In [ ]:
# Cell 5: 結果読込 + 可視化
import json
import numpy as np
import matplotlib.pyplot as plt

with open('phase_c_v3_results.json') as f:
    results = json.load(f)

# 条件間比較テーブル
print('='*90)
print('  Phase C v3 — 3条件アブレーション')
print('='*90)
print(f"{'条件':<6} {'λ':<10} {'ρ_49d':>7} {'ρ_ccl':>7} {'偏ρ':>7} {'R@1':>5} {'R@5':>5} {'R@10':>5} {'MSE':>7}")
print('-'*90)
print(f"{'C-mini':<6} {'ref':<10} {'0.963':>7} {'—':>7} {'0.960':>7} {'1.00':>5} {'—':>5} {'—':>5} {'0.010':>7}")

for cond_data in results['conditions']:
    cond = cond_data['condition']
    n = cond_data['n_pairs']
    for lxi_name, data in cond_data['results'].items():
        print(f"{cond:<6} {lxi_name:<10} "
              f"{data['mean_rho']:>7.4f} {data.get('mean_rho_ccl', 0):>7.4f} {data['mean_partial_rho']:>7.4f} "
              f"{data['mean_r_at_1']:>5.2f} {data['mean_r_at_5']:>5.2f} {data.get('mean_r_at_10', 0):>5.2f} "
              f"{data['mean_mse']:>7.4f}")

# 可視化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

conds = [c['condition'] for c in results['conditions']]
rho_49d = [c['results']['baseline']['mean_rho'] for c in results['conditions']]
rho_ccl = [c['results']['baseline'].get('mean_rho_ccl', 0) for c in results['conditions']]

x = np.arange(len(conds))
w = 0.3
axes[0].bar(x - w/2, rho_49d, w, label='ρ_49d', color='steelblue')
axes[0].bar(x + w/2, rho_ccl, w, label='ρ_ccl', color='coral')
axes[0].axhline(y=0.963, color='red', linestyle='--', alpha=0.5, label='C-mini')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'Cond {c}' for c in conds])
axes[0].set_ylabel('Spearman ρ')
axes[0].set_title('P42: ρ_49d vs ρ_ccl (baseline)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# L_Ξ ablation
for cd in results['conditions']:
    lxi_names = list(cd['results'].keys())
    rhos = [cd['results'][k]['mean_rho'] for k in lxi_names]
    axes[1].plot(range(len(lxi_names)), rhos, 'o-', label=f"Cond {cd['condition']}")
axes[1].axhline(y=0.963, color='red', linestyle='--', alpha=0.5)
axes[1].set_xticks(range(len(lxi_names)))
axes[1].set_xticklabels(lxi_names, rotation=45)
axes[1].set_title('P14: L_Ξ Ablation')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# R@k 比較
r1s = [c['results']['baseline']['mean_r_at_1'] for c in results['conditions']]
r5s = [c['results']['baseline']['mean_r_at_5'] for c in results['conditions']]
r10s = [c['results']['baseline'].get('mean_r_at_10', 0) for c in results['conditions']]
axes[2].bar(x - w, r1s, w, label='R@1')
axes[2].bar(x, r5s, w, label='R@5')
axes[2].bar(x + w, r10s, w, label='R@10')
axes[2].set_xticks(x)
axes[2].set_xticklabels([f'Cond {c}' for c in conds])
axes[2].set_title('Retrieval Metrics')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('phase_c_v3_results.png', dpi=150)
plt.show()

In [ ]:
# Cell 6: ダウンロード
files.download('phase_c_v3_results.json')
files.download('phase_c_v3_results.png')